# Feature Engineering

**Project:** Energy Forecasting — Household Electricity Consumption (Enel × LUISS).

**Goal of this notebook:** apply the preprocessing and feature pipelines to the real UCI dataset, produce a model-ready feature matrix `X` and target `y`, and persist them to disk as the input for the modeling notebook. Every feature appearing in `X` must be leakage-safe per the registry — the pipeline refuses to emit anything else in strict mode (see [`tests/integration/test_leakage.py`](../tests/integration/test_leakage.py) for the locked-in guarantees).

**Reading order.** Each section runs a discrete pipeline stage on the real data, inspects the output, then interprets what we see and what it implies for downstream modeling. The handoff table from EDA — [`reports/results/eda_decisions.csv`](../reports/results/eda_decisions.csv) — is the source of truth for every feature decision made here.

---

## 1 · Setup

Same opening as the EDA notebook: import the package, load the pinned config, fix the random seed. Confirm we are running the strict leakage-safe policy.

In [1]:
# =============================================================
# 1.1  IMPORTS AND CONFIGURATION
# =============================================================
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from energy_forecasting.config import DEFAULT_CONFIG_PATH, ForecastConfig
from energy_forecasting.data import load_raw
from energy_forecasting.preprocessing import preprocess
from energy_forecasting.features import FEATURE_REGISTRY, build_features, is_safe
from energy_forecasting.utils import set_global_seed

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110

cfg = ForecastConfig.from_yaml(DEFAULT_CONFIG_PATH)
set_global_seed(cfg.reproducibility.seed)

print(f"Config hash         : {cfg.content_hash()[:12]}")
print(f"Target              : {cfg.target.column}")
print(f"Frequency / horizon : {cfg.target.frequency}  /  {cfg.target.horizon_hours}h")
print(f"Strict leakage-safe : {cfg.features.strict_leakage_safe}")
print(f"Registry entries    : {len(FEATURE_REGISTRY)}")
print(f"Lag hours           : {list(cfg.features.lag_hours)}")
print(f"Rolling windows     : {list(cfg.features.rolling_window_hours)}")

Config hash         : 050755a0a6eb
Target              : Global_active_power
Frequency / horizon : 1h  /  24h
Strict leakage-safe : True
Registry entries    : 29
Lag hours           : [1, 24, 168]
Rolling windows     : [3, 24, 168]


## 2 · Preprocessing — From Raw to Hourly

Re-run the same pipeline the EDA notebook used: `load_raw → preprocess`. We don't re-derive the policy here, just apply the EDA-validated one. The returned `PreprocessingReport` is the audit trail for what happened, captured for reporting.

In [2]:
# =============================================================
# 2.1  RAW → HOURLY PIPELINE
# =============================================================
raw_path = Path("..") / "data" / "raw" / "data.txt"
df_raw = load_raw(raw_path)
df_hourly, prep_report = preprocess(df_raw, config=cfg)

print("Preprocessing report:")
for field_name in [
    "input_rows", "output_rows", "target_frequency",
    "n_total_gaps", "n_short_gaps_filled", "n_long_gaps_preserved",
    "longest_gap_rows", "rows_with_residual_nan",
]:
    v = getattr(prep_report, field_name)
    fmt = f"{v:,}" if isinstance(v, int) else str(v)
    print(f"  {field_name:24s}: {fmt}")

print()
print("Hourly DataFrame columns:")
for c in df_hourly.columns:
    safety = is_safe(c)
    flag = "safe" if safety is True else ("UNSAFE" if safety is False else "unknown")
    print(f"  {c:30s}  [{flag}]")

2026-05-14 15:39:07 | INFO     | energy_forecasting.data.loader:load_raw:71 | Loading raw dataset from ..\data\raw\data.txt


2026-05-14 15:39:48 | INFO     | energy_forecasting.data.loader:load_raw:117 | Loaded 2,075,259 rows × 7 columns, range 2006-12-16 17:24:00 → 2010-11-26 21:02:00


2026-05-14 15:39:54 | INFO     | energy_forecasting.preprocessing.missing:apply_missing_value_policy:154 | Missing-value policy applied: 25,979 originally missing rows; 54 short gaps interpolated (72 rows filled), 17 long gaps preserved (25,907 rows remain NaN)


2026-05-14 15:39:55 | INFO     | energy_forecasting.preprocessing.resampling:resample_to_frequency:97 | Dropped 421 empty buckets at frequency '1h'


2026-05-14 15:39:55 | INFO     | energy_forecasting.preprocessing.resampling:resample_to_frequency:100 | Resampled to '1h': 2,075,259 rows -> 34,168 rows, aggregation='mean'


2026-05-14 15:39:55 | INFO     | energy_forecasting.preprocessing.pipeline:preprocess:125 | Preprocessing complete: 2,075,259 raw rows -> 34,168 rows at 1h, 0 rows still NaN (long outages)


Preprocessing report:
  input_rows              : 2,075,259
  output_rows             : 34,168
  target_frequency        : 1h
  n_total_gaps            : 71
  n_short_gaps_filled     : 54
  n_long_gaps_preserved   : 17
  longest_gap_rows        : 7,226
  rows_with_residual_nan  : 0

Hourly DataFrame columns:
  Global_active_power             [safe]
  Global_reactive_power           [UNSAFE]
  Voltage                         [UNSAFE]
  Global_intensity                [UNSAFE]
  Sub_metering_1                  [UNSAFE]
  Sub_metering_2                  [UNSAFE]
  Sub_metering_3                  [UNSAFE]
  is_originally_missing           [safe]
  is_outage_gap                   [safe]


## 3 · Build the Feature Matrix

Call [`build_features`](../src/energy_forecasting/features/pipeline.py) with the same config that governs everything else. The pipeline:

1. Extracts the target series (`Global_active_power`) as `y`.
2. Drops every input column registered as `leakage_safe=False` (the contemporaneous physical measurements).
3. Builds calendar + cyclical + lag + rolling + holiday features per the config.
4. Verifies every output column is registered-safe — raises `LeakageError` if any unsafe column would be emitted.
5. Drops the warm-up rows where lag-168 / rolling-168 features are NaN.

We then inspect the result before any modeling decisions are made.

In [3]:
# =============================================================
# 3.1  BUILD THE LEAKAGE-SAFE FEATURE MATRIX
# =============================================================
fm = build_features(df_hourly, config=cfg)

print(f"X shape                  : {fm.X.shape}")
print(f"y shape                  : {fm.y.shape}")
print(f"Unsafe inputs dropped    : {fm.n_unsafe_dropped}")
print(f"Rows before dropna       : {fm.n_rows_before_dropna:,}")
print(f"Rows after dropna        : {fm.n_rows_after_dropna:,}")
print(f"Retention rate           : {fm.n_rows_after_dropna / fm.n_rows_before_dropna * 100:.2f}%")
print(f"Date range (X.index)     : {fm.X.index.min()}  →  {fm.X.index.max()}")
print(f"y descriptive stats      :")
for stat, val in fm.y.describe().round(3).items():
    print(f"  {stat:>5s}: {val}")

2026-05-14 15:39:55 | INFO     | energy_forecasting.features.pipeline:build_features:97 | Dropping 6 unsafe input columns: ['Global_reactive_power', 'Voltage', 'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']


2026-05-14 15:39:56 | INFO     | energy_forecasting.features.holidays_feature:build_french_holiday_feature:57 | French holidays: 1,008 of 34,168 timestamps fall on 42 distinct holiday dates


2026-05-14 15:39:56 | INFO     | energy_forecasting.features.pipeline:build_features:167 | Feature matrix built: 31 features, 34,000 rows after warm-up dropna (was 34,168, retained 99.51%)


X shape                  : (34000, 31)
y shape                  : (34000,)
Unsafe inputs dropped    : 6
Rows before dropna       : 34,168
Rows after dropna        : 34,000
Retention rate           : 99.51%
Date range (X.index)     : 2006-12-23 17:00:00  →  2010-11-26 21:00:00
y descriptive stats      :
  count: 34000.0
   mean: 1.088
    std: 0.895
    min: 0.124
    25%: 0.342
    50%: 0.8
    75%: 1.575
    max: 6.561


In [4]:
# =============================================================
# 3.2  FEATURE CATALOG — WHAT WAS BUILT, BY CATEGORY
# =============================================================
from collections import defaultdict

by_category: dict[str, list[str]] = defaultdict(list)
for col in fm.X.columns:
    # Find the registry entry (exact match, then wildcard prefix)
    spec = FEATURE_REGISTRY.get(col)
    if spec is None:
        for pat, s in FEATURE_REGISTRY.items():
            if pat.endswith("*") and col.startswith(pat[:-1]):
                spec = s
                break
    by_category[spec.category if spec else "unknown"].append(col)

for cat in sorted(by_category):
    print(f"[{cat}]  ({len(by_category[cat])} features)")
    for name in by_category[cat]:
        print(f"    {name}")
    print()

[calendar]  (7 features)
    hour
    day_of_week
    day_of_month
    day_of_year
    month
    quarter
    is_weekend

[cyclical]  (6 features)
    hour_sin
    hour_cos
    day_of_week_sin
    day_of_week_cos
    month_sin
    month_cos

[external]  (1 features)
    is_french_holiday

[indicator]  (2 features)
    is_originally_missing
    is_outage_gap

[lag]  (3 features)
    Global_active_power_lag_1h
    Global_active_power_lag_24h
    Global_active_power_lag_168h

[rolling]  (12 features)
    Global_active_power_roll_mean_3h
    Global_active_power_roll_std_3h
    Global_active_power_roll_min_3h
    Global_active_power_roll_max_3h
    Global_active_power_roll_mean_24h
    Global_active_power_roll_std_24h
    Global_active_power_roll_min_24h
    Global_active_power_roll_max_24h
    Global_active_power_roll_mean_168h
    Global_active_power_roll_std_168h
    Global_active_power_roll_min_168h
    Global_active_power_roll_max_168h



## 4 · Leakage Policy In Action

The single most important architectural property of this pipeline is that it cannot accidentally re-introduce the leakage problem this kind of code could otherwise reintroduce. This section demonstrates the policy *firing live*, so the reader of the report can see that "strict leakage-safe" is not a label — it is enforced code.

Two demos:

- **4.1** Show the registry's classification of every contemporaneous raw column as unsafe.
- **4.2** Attempt to inject an unsafe feature into the output matrix and observe that the pipeline refuses it, raising `LeakageError`.

### 4.1 Registry classification of contemporaneous raw columns

The registry already declares every column in [`features/registry.py`](../src/energy_forecasting/features/registry.py) with a written rationale for why it is or isn't safe to use as a model feature. The table below confirms that all six columns an earlier draft of this pipeline used as features are now flagged unsafe — the rationale links each one to the physical/structural reason it would leak.

In [5]:
# =============================================================
# 4.1  REGISTRY CLASSIFICATION FOR THE ORIGINAL LEAKED COLUMNS
# =============================================================
leaked_in_original = [
    "Global_intensity",   # P = V × I — direct target leakage via Ohm's law
    "Voltage",            # contemporaneous with target, unknown at forecast time
    "Global_reactive_power",  # contemporaneous, tightly correlated via power factor
    "Sub_metering_1",     # kitchen sub-meter — component of target
    "Sub_metering_2",     # laundry sub-meter — component of target
    "Sub_metering_3",     # climate sub-meter — component of target
]

print(f"{'Column':<25} {'Registry leakage_safe':<22} {'Why':<60}")
print("-" * 110)
for col in leaked_in_original:
    spec = FEATURE_REGISTRY[col]
    safe_str = str(spec.leakage_safe)
    print(f"{col:<25} {safe_str:<22} {spec.description}")

Column                    Registry leakage_safe  Why                                                         
--------------------------------------------------------------------------------------------------------------
Global_intensity          False                  Current intensity at the same timestamp as target. P = V × I gives the model the answer directly.
Voltage                   False                  Same-timestamp voltage. Physically related to active power through the load's power factor; unknown at forecast time.
Global_reactive_power     False                  Same-timestamp reactive power. Tightly correlated with active power via the household's overall power factor.
Sub_metering_1            False                  Kitchen sub-meter at same timestamp. Component of Global_active_power.
Sub_metering_2            False                  Laundry sub-meter at same timestamp. Component of Global_active_power.
Sub_metering_3            False                  Climate sub-meter

### 4.2 Attempting to inject an unsafe feature

Construct an `X_corrupted` matrix that contains a column known to be unsafe (`Voltage`). The leakage check in [`build_features`](../src/energy_forecasting/features/pipeline.py) runs on the pipeline output — but to demonstrate the *check itself* without re-running the whole pipeline, we call the registry's `is_safe` lookup directly on each column. Then we exhibit what would happen if `strict_leakage_safe=True` (the default) saw an unsafe column.

In [6]:
# =============================================================
# 4.2  WHAT HAPPENS IF AN UNSAFE FEATURE SLIPS THROUGH
# =============================================================
from energy_forecasting.exceptions import LeakageError

# Construct a hypothetical corrupted feature matrix by adding the
# contemporaneous Voltage column back to X. (Note: this is a deliberate
# attempt to demonstrate the guard — not something the pipeline would do.)
X_corrupted = fm.X.join(df_hourly["Voltage"], how="inner")
print(f"Corrupted X shape: {X_corrupted.shape}")
print(f"Last column added: {X_corrupted.columns[-1]}")

# Now run the same leakage check the pipeline runs on its output.
unsafe = [c for c in X_corrupted.columns if is_safe(c) is False]
unknown = [c for c in X_corrupted.columns if is_safe(c) is None]

print()
print(f"Unsafe columns detected   : {unsafe}")
print(f"Unknown columns detected  : {unknown}")

if cfg.features.strict_leakage_safe and (unsafe or unknown):
    try:
        raise LeakageError(
            f"Leakage-unsafe columns in feature matrix: {unsafe}. "
            f"This is exactly the error the pipeline raises when strict_leakage_safe=True."
        )
    except LeakageError as e:
        print()
        print("GUARD FIRED — LeakageError raised as expected:")
        print(f"  {e}")

Corrupted X shape: (34000, 32)
Last column added: Voltage

Unsafe columns detected   : ['Voltage']
Unknown columns detected  : []

GUARD FIRED — LeakageError raised as expected:
  Leakage-unsafe columns in feature matrix: ['Voltage']. This is exactly the error the pipeline raises when strict_leakage_safe=True.


**Interpretation.** Two important properties demonstrated:

1. **The registry knows about every dangerous column.** All six contemporaneous raw measurements that produced the suspicious MAE = 0.0136 in an earlier draft of this pipeline are explicitly flagged `leakage_safe=False` with a written rationale. The rationales are short and reviewable.
2. **The check fires loud, not quiet.** When an unsafe column appears in the would-be feature matrix, the system raises `LeakageError` with a clear message — not a warning, not a silent skip. There is no "forgot to filter" failure mode in strict mode.

**Implication.** Any future contributor (or future-me) who tries to add `Voltage` as a feature "because it correlates strongly with the target" will be stopped at the feature pipeline. The only way past the guard is to explicitly set `features.strict_leakage_safe: false` in [`conf/base.yaml`](../conf/base.yaml), and that change is reviewable in git. This is what "leakage prevention by construction" means.

## 5 · Inspect and Persist the Feature Matrix

Before saving the artifact, look at the actual shape of the data — the first few rows, dtypes, NaN check. Then write `X` and `y` to [`data/processed/features.parquet`](../data/processed/features.parquet) along with a tiny metadata sidecar so the modeling notebook can verify it's loading the artifact built from the same config.

In [7]:
# =============================================================
# 5.1  HEAD AND DTYPE AUDIT
# =============================================================
# Show a window of rows (well past the lag-168 warm-up so all features are populated).
display(fm.X.head(3).round(3))

print()
print("Dtype distribution:")
dtype_counts = fm.X.dtypes.value_counts()
for d, n in dtype_counts.items():
    print(f"  {str(d):>10s}: {n} columns")

n_nan_rows = int(fm.X.isna().any(axis=1).sum())
n_nan_y = int(fm.y.isna().sum())
print()
print(f"Rows with any NaN in X : {n_nan_rows}")
print(f"NaN values in y        : {n_nan_y}")
assert n_nan_rows == 0 and n_nan_y == 0, "Unexpected NaN — modeling expects fully populated matrix"

,is_originally_missing,is_outage_gap,hour,day_of_week,day_of_month,day_of_year,month,quarter,is_weekend,hour_sin,...,Global_active_power_roll_max_3h,Global_active_power_roll_mean_24h,Global_active_power_roll_std_24h,Global_active_power_roll_min_24h,Global_active_power_roll_max_24h,Global_active_power_roll_mean_168h,Global_active_power_roll_std_168h,Global_active_power_roll_min_168h,Global_active_power_roll_max_168h,is_french_holiday
datetime,,,,,,,,,,,,,,,,,,,,,
2006-12-23 17:00:00,0,0,17,5,23,357,12,4,1,-0.966,...,4.349,2.935,0.990,1.497,4.549,1.764,1.146,0.247,4.549,0
2006-12-23 18:00:00,0,0,18,5,23,357,12,4,1,-1.000,...,5.453,3.100,1.067,1.649,5.453,1.771,1.166,0.247,5.453,0
2006-12-23 19:00:00,0,0,19,5,23,357,12,4,1,-0.966,...,5.453,3.149,1.074,1.649,5.453,1.773,1.168,0.247,5.453,0



Dtype distribution:
     float64: 15 columns
        int8: 9 columns
     float32: 6 columns
       int16: 1 columns

Rows with any NaN in X : 0
NaN values in y        : 0


In [8]:
# =============================================================
# 5.2  PERSIST FEATURE MATRIX + METADATA SIDECAR
# =============================================================
import json
from datetime import datetime, timezone

out_dir = Path("..") / "data" / "processed"
out_dir.mkdir(parents=True, exist_ok=True)
features_path = out_dir / "features.parquet"
metadata_path = out_dir / "features.metadata.json"

# Combine X and y into a single parquet (target as its own column).
combined = fm.X.copy()
combined[cfg.target.column] = fm.y
combined.to_parquet(features_path, compression="snappy")

# Sidecar metadata — config hash + provenance for downstream verification.
metadata = {
    "config_hash": cfg.content_hash(),
    "target_column": cfg.target.column,
    "target_frequency": cfg.target.frequency,
    "n_rows": int(len(fm.X)),
    "n_features": int(fm.X.shape[1]),
    "feature_names": list(fm.feature_names),
    "date_range_start": str(fm.X.index.min()),
    "date_range_end": str(fm.X.index.max()),
    "unsafe_inputs_dropped": int(fm.n_unsafe_dropped),
    "strict_leakage_safe": bool(cfg.features.strict_leakage_safe),
    "built_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
}
metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

print(f"Saved features  : {features_path.resolve()}")
print(f"Saved metadata  : {metadata_path.resolve()}")
print(f"File size (MB)  : {features_path.stat().st_size / 1e6:.2f}")
print()
print("Metadata sidecar contents:")
print(json.dumps(metadata, indent=2)[:1200])  # truncate feature list for readability

Saved features  : C:\Users\THIERRY\OneDrive\Desktop\AI\energy-forecasting\data\processed\features.parquet
Saved metadata  : C:\Users\THIERRY\OneDrive\Desktop\AI\energy-forecasting\data\processed\features.metadata.json
File size (MB)  : 3.79

Metadata sidecar contents:
{
  "config_hash": "050755a0a6eb75004393efb77ad523d0d04677715669665313ddb5d4efeb4b9a",
  "target_column": "Global_active_power",
  "target_frequency": "1h",
  "n_rows": 34000,
  "n_features": 31,
  "feature_names": [
    "is_originally_missing",
    "is_outage_gap",
    "hour",
    "day_of_week",
    "day_of_month",
    "day_of_year",
    "month",
    "quarter",
    "is_weekend",
    "hour_sin",
    "hour_cos",
    "day_of_week_sin",
    "day_of_week_cos",
    "month_sin",
    "month_cos",
    "Global_active_power_lag_1h",
    "Global_active_power_lag_24h",
    "Global_active_power_lag_168h",
    "Global_active_power_roll_mean_3h",
    "Global_active_power_roll_std_3h",
    "Global_active_power_roll_min_3h",
    "Global_a

## 6 · Handoff to Modeling

What this notebook produced and what comes next.

In [9]:
# =============================================================
# 6.1  HANDOFF SUMMARY
# =============================================================
handoff_summary = pd.DataFrame([
    {"item": "Artifact",          "value": str(features_path.resolve())},
    {"item": "Metadata sidecar",  "value": str(metadata_path.resolve())},
    {"item": "Config hash",       "value": cfg.content_hash()[:16] + "…"},
    {"item": "Rows",              "value": f"{len(fm.X):,}"},
    {"item": "Features",          "value": f"{fm.X.shape[1]} (all leakage-safe)"},
    {"item": "Target column",     "value": cfg.target.column},
    {"item": "Frequency",         "value": cfg.target.frequency},
    {"item": "Date range",        "value": f"{fm.X.index.min()} → {fm.X.index.max()}"},
    {"item": "Strict leakage",    "value": str(cfg.features.strict_leakage_safe)},
    {"item": "Next notebook",     "value": "03_modeling.ipynb (Phase 4 in plan)"},
])
display(handoff_summary)

,item,value
0,Artifact,C:\Users\THIERRY\OneDrive\Desktop\AI\energy-fo...
1,Metadata sidecar,C:\Users\THIERRY\OneDrive\Desktop\AI\energy-fo...
2,Config hash,050755a0a6eb7500…
3,Rows,"34,000"
4,Features,31 (all leakage-safe)
5,Target column,Global_active_power
6,Frequency,1h
7,Date range,2006-12-23 17:00:00 → 2010-11-26 21:00:00
8,Strict leakage,True
9,Next notebook,03_modeling.ipynb (Phase 4 in plan)


---

**Closing reflection.**

Two things this notebook locked in for downstream:

1. **The feature matrix is one canonical artifact.** Anyone modeling on this dataset reads [`data/processed/features.parquet`](../data/processed/features.parquet) and trusts it — they don't re-derive features ad-hoc. The metadata sidecar lets the modeling notebook assert `metadata['config_hash'] == cfg.content_hash()` and refuse to run if they disagree.
2. **The leakage policy is provably active.** §4 demonstrated the registry's classification of every dangerous column and showed the guard firing on an injection attempt. The same guarantee is locked in by [`tests/integration/test_leakage.py`](../tests/integration/test_leakage.py) — those tests run in CI on every commit.

**Next.** The modeling notebook (Phase 4 in the plan) will: (a) read this artifact and its metadata, (b) apply the rolling-origin split plan from the splitter module (Phase 3A), (c) fit and evaluate the model lineup (Phase 4), (d) produce the leaderboard required by the project brief.